# 🎙️ PrecisionVoice - Vietnamese Speech-to-Text

Notebook đơn giản để transcribe audio tiếng Việt sử dụng **faster-whisper** và **pyannote** (diarization).

### Hướng dẫn
1. **Chọn GPU**: `Runtime` → `Change runtime type` → **T4 GPU**
2. **Cài đặt Secrets**: Thêm `HF_TOKEN` vào Colab Secrets (Key icon bên trái) để dùng Pyannote.
3. **Chạy từng cell** theo thứ tự từ trên xuống
4. **Sử dụng Gradio link** ở cell cuối để truy cập UI

In [ ]:
# 1. (Đã gộp) Cài đặt Dependencies ở bước 2 để tránh xung đột phiên bản.


In [ ]:
# @title 1. 🔍 Kiểm tra GPU
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Detected: {gpu_name}")
    print(f"   VRAM: {gpu_mem:.1f} GB")
else:
    print("⚠️ KHÔNG TÌM THẤY GPU!")
    print("👉 Vào Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title 2. 📦 Cài đặt Dependencies
print("Installing dependencies...")
!pip install --upgrade torch torchvision torchaudio "pyannote.audio>=3.3.1" faster-whisper gradio librosa nest_asyncio lightning torchmetrics
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ Dependencies installed successfully!")

In [ ]:
# @title 3. 🤖 Load Models (Whisper & Pyannote)
import torch
import time
import os
from google.colab import userdata
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
try:
    from pyannote.audio.core.task import Specifications, Problem, Resolution
    torch.serialization.add_safe_globals([Specifications, Problem, Resolution])
except Exception as e:
    print(f"Could not add custom globals: {e}")

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Whisper
print("Loading EraX-WoW-Turbo model (optimized for Vietnamese)...")
start = time.time()
model = WhisperModel(
    "erax-ai/EraX-WoW-Turbo-V1.1-CT2",
    device=device,
    compute_type="float16" if device == "cuda" else "int8"
)
print(f"✅ Whisper loaded in {time.time() - start:.1f}s")

# 2. Load Pyannote
print("Loading Pyannote Diarization...")
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = os.environ.get('HF_TOKEN')

diarization_pipeline = None
if not hf_token:
    print("⚠️ WARNING: HF_TOKEN not found! Diarization might fail.")
    print("Please set HF_TOKEN in Colab Secrets.")
else:
    start = time.time()
    try:
        diarization_pipeline = Pipeline.from_pretrained(
            "pyannote/speaker-diarization-community-1",
            token=hf_token
        )
        diarization_pipeline.to(torch.device(device))
        print(f"✅ Pyannote loaded in {time.time() - start:.1f}s")
    except Exception as e:
        print(f"❌ Failed to load Pyannote: {e}")


In [ ]:
# @title 4. 🎤 Khởi chạy Gradio UI
import gradio as gr
import time
import torch
import nest_asyncio
import os

nest_asyncio.apply()

def format_timestamp(seconds):
    minutes = int(seconds // 60)
    seconds = seconds % 60
    return f"{minutes:02d}:{seconds:05.2f}"

def transcribe(audio_path, language, beam_size, vad_filter, p=gr.Progress()):
    if audio_path is None:
        yield "⚠️ Vui lòng upload hoặc ghi âm audio!"
        return
    
    start_time = time.time()
    
    # 1. Transcribe
    p(0.1, desc="1/3: Đang Whisper transcription...")
    segments_gen, info = model.transcribe(
        audio_path, language=language if language != "auto" else None,
        beam_size=beam_size, vad_filter=vad_filter, word_timestamps=True
    )
    
    whisper_segments = list(segments_gen)
    p(0.5, desc="2/3: Đang thực hiện Diarization...")
    
    # 2. Diarize
    diarization_result = None
    if diarization_pipeline:
        try:
            # Pyannote expects a path
            diarization_result = diarization_pipeline(audio_path)
        except Exception as e:
            print(f"Diarization error: {e}")
    
    # 3. Merge
    p(0.8, desc="3/3: Đang tổng hợp kết quả...")
    final_output = []
    detected_speakers = set()
    
    for seg in whisper_segments:
        text = seg.text
        start = seg.start
        end = seg.end
        
        speaker = "Unknown"
        if diarization_result:
            # Find overlap
            best_speaker = None
            max_overlap = 0
            for turn, _, spk in diarization_result.speaker_diarization.itertracks(yield_label=True):
                # Intersect
                s_inter = max(start, turn.start)
                e_inter = min(end, turn.end)
                overlap = max(0, e_inter - s_inter)
                if overlap > max_overlap:
                    max_overlap = overlap
                    best_speaker = spk
            if best_speaker:
                speaker = best_speaker
        
        detected_speakers.add(speaker)
        final_output.append(f"[{format_timestamp(start)} -> {format_timestamp(end)}] **{speaker}**: {text}")
    
    elapsed = time.time() - start_time
    output = f"📊 Ngôn ngữ: {info.language} ({info.language_probability:.1%})\n"
    output += f"👥 Số người nói phát hiện: {len(detected_speakers)}\n"
    output += f"⏱️ Thời gian xử lý: {elapsed:.1f}s\n"
    output += f"━" * 50 + "\n\n"
    output += "\n".join(final_output)
    
    yield output

with gr.Blocks(title="PrecisionVoice", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ PrecisionVoice - Vietnamese STT (Whisper + Pyannote)")
    gr.Markdown("Sử dụng **EraX-WoW-Turbo** để nhận dạng văn bản và **Pyannote Community** để phân biệt người nói.")

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(sources=["upload", "microphone"], type="filepath", label="🔊 Audio Input")
            language = gr.Dropdown(choices=["auto", "vi", "en"], value="vi", label="🌐 Ngôn ngữ")
            with gr.Accordion("Cài đặt", open=False):
                beam_size = gr.Slider(minimum=1, maximum=10, value=5, label="Beam Size")
                vad_filter = gr.Checkbox(value=True, label="VAD Filter")
            btn = gr.Button("🚀 Bắt đầu", variant="primary")
        
        with gr.Column(scale=2):
            output = gr.Markdown(label="Kết quả")
            
    btn.click(transcribe, inputs=[audio_input, language, beam_size, vad_filter], outputs=[output])

import os
if "COLAB_GPU" in os.environ or "google.colab" in str(get_ipython()):
    demo.queue().launch(share=True, debug=True)
else:
    demo.launch(share=False)
